# 01 — EDA: NSE Stocks (12-Ticker Universe)

> **Portfolio Optimizer** · *Exploratory Data Analysis*
>
> Palette: `#1A273A` · `#3E4A62` · `#C24D2C` · `#D9D9D7`

**Objectives**
1. Pull 3 years of OHLCV data for all 12 NSE tickers via yfinance.
2. Compute descriptive statistics, return distributions, and correlation structure.
3. Visualise price history, rolling volatility, and drawdowns.
4. Profile seasonal and intraday patterns (NSE market hours).
5. Identify data quality issues (missing bars, outliers, splits).

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# ── Design Tokens ────────────────────────────────────────────────────────────
NAVY    = '#1A273A'
SLATE   = '#3E4A62'
ORANGE  = '#C24D2C'
GREY    = '#D9D9D7'

def base_layout(**kwargs):
    return dict(
        paper_bgcolor=NAVY, plot_bgcolor=NAVY,
        font=dict(color=GREY, family='Inter, sans-serif'),
        xaxis=dict(gridcolor=SLATE, color=GREY),
        yaxis=dict(gridcolor=SLATE, color=GREY),
        **kwargs,
    )

# ── Constants ────────────────────────────────────────────────────────────────
TICKERS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS',
    'ICICIBANK.NS', 'WIPRO.NS', 'BAJFINANCE.NS', 'ASIANPAINT.NS',
    'TITAN.NS', 'MARUTI.NS', 'ONGC.NS', 'ZOMATO.NS',
]
START = '2020-01-01'
END   = '2023-12-31'
print(f'Tickers: {len(TICKERS)}  |  Period: {START} → {END}')

In [ ]:
# ── 1. Data Download ──────────────────────────────────────────────────────────
df_raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=True)
print(f'Shape: {df_raw.shape}  |  Columns: {df_raw.columns.get_level_values(0).unique().tolist()}')
df_raw.head(3)

In [ ]:
# ── 2. Missing Data Audit ─────────────────────────────────────────────────────
close = df_raw['Close']
missing = close.isnull().sum()
pct_missing = (close.isnull().mean() * 100).round(2)

fig = go.Figure(go.Bar(
    x=TICKERS, y=pct_missing.values,
    marker_color=ORANGE, text=pct_missing.values.round(1),
    textposition='outside',
))
fig.update_layout(**base_layout(title='Missing Data % per Ticker', height=400))
fig.show()

print(f'Max missing: {pct_missing.max():.1f}%  ({pct_missing.idxmax()})')

In [ ]:
# ── 3. Normalised Price History ────────────────────────────────────────────────
norm = (close / close.iloc[0] * 100).ffill()

fig = go.Figure()
colours = px.colors.qualitative.Safe
for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=norm.index, y=norm[ticker],
        name=ticker.replace('.NS', ''),
        line=dict(color=colours[i % len(colours)], width=1.5),
    ))

fig.update_layout(**base_layout(title='Rebased Price (Jan 2020 = 100)', height=550,
                                  hovermode='x unified'))
fig.show()

In [ ]:
# ── 4. Return Distribution ────────────────────────────────────────────────────
returns = close.pct_change().dropna()

fig = make_subplots(rows=3, cols=4,
    subplot_titles=[t.replace('.NS','') for t in TICKERS])

for i, ticker in enumerate(TICKERS):
    r, c = divmod(i, 4)
    data = returns[ticker].dropna()
    fig.add_trace(
        go.Histogram(x=data, nbinsx=80, marker_color=ORANGE,
                     name=ticker.replace('.NS',''), showlegend=False),
        row=r+1, col=c+1,
    )

fig.update_layout(**base_layout(title='Daily Return Distributions', height=700))
fig.update_annotations(font_color=GREY)
fig.show()

# Summary stats
stats = returns.describe().T
stats['skew']     = returns.skew()
stats['kurtosis'] = returns.kurtosis()
stats['ann_vol']  = returns.std() * np.sqrt(252)
print('\nReturn Statistics:')
stats[['mean','std','skew','kurtosis','ann_vol']].round(4)

In [ ]:
# ── 5. Correlation Matrix ─────────────────────────────────────────────────────
corr = returns.corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns.str.replace('.NS',''),
    y=corr.index.str.replace('.NS',''),
    colorscale=[[0, NAVY], [0.5, SLATE], [1, ORANGE]],
    zmin=-1, zmax=1, text=corr.round(2).values, texttemplate='%{text}',
))
fig.update_layout(**base_layout(title='Return Correlation Matrix (2020–2023)', height=550))
fig.show()

# Highly correlated pairs
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
           .stack().sort_values(ascending=False))
print('Top 5 correlated pairs:')
print(pairs.head(5).to_string())

In [ ]:
# ── 6. Rolling Volatility (30-day) ────────────────────────────────────────────
roll_vol = returns.rolling(30).std() * np.sqrt(252) * 100

fig = go.Figure()
for i, ticker in enumerate(TICKERS[:5]):  # top 5 for clarity
    fig.add_trace(go.Scatter(
        x=roll_vol.index, y=roll_vol[ticker],
        name=ticker.replace('.NS',''),
        line=dict(color=colours[i % len(colours)], width=1.5),
    ))

# COVID crash band
fig.add_vrect(x0='2020-02-01', x1='2020-05-01',
              fillcolor=ORANGE, opacity=0.15, annotation_text='COVID-19 Crash')

fig.update_layout(**base_layout(title='30-Day Rolling Annualised Volatility (%)', height=450))
fig.show()

In [ ]:
# ── 7. Maximum Drawdown ───────────────────────────────────────────────────────
def max_drawdown_series(prices):
    roll_max = prices.cummax()
    dd_series = (prices - roll_max) / roll_max * 100
    return dd_series

fig = go.Figure()
for i, ticker in enumerate(TICKERS[:6]):
    dd = max_drawdown_series(close[ticker].ffill())
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd,
        name=ticker.replace('.NS',''),
        fill='tozeroy', fillcolor=f'rgba{tuple(int(colours[i%len(colours)].lstrip("#")[j:j+2],16) for j in (0,2,4)) + (0.2,)}',
        line=dict(color=colours[i % len(colours)], width=1),
    ))

fig.update_layout(**base_layout(title='Drawdown from ATH (%)', height=450))
fig.show()

# Max drawdown table
dd_stats = pd.DataFrame({'Max Drawdown (%)': {t: max_drawdown_series(close[t].ffill()).min() for t in TICKERS}})
print(dd_stats.sort_values('Max Drawdown (%)').round(2))

In [ ]:
# ── 8. Volume Analysis ────────────────────────────────────────────────────────
volume = df_raw['Volume'].ffill()
avg_vol = volume.mean().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=avg_vol.index.str.replace('.NS',''), y=avg_vol.values / 1e6,
    marker_color=[ORANGE if t == avg_vol.index[0] else SLATE for t in avg_vol.index],
    text=(avg_vol.values / 1e6).round(1), textposition='outside',
))
fig.update_layout(**base_layout(title='Average Daily Volume (Millions)', height=400,
                                  yaxis_title='Volume (M shares)'))
fig.show()

In [ ]:
# ── 9. Monthly Return Heatmap — TCS ──────────────────────────────────────────
ticker_for_heatmap = 'TCS.NS'
monthly_ret = (
    returns[ticker_for_heatmap]
        .resample('ME').apply(lambda x: (1 + x).prod() - 1) * 100
)
pivot = (pd.DataFrame({'ret': monthly_ret.values, 'year': monthly_ret.index.year,
                        'month': monthly_ret.index.strftime('%b')})
           .pivot(index='year', columns='month', values='ret'))

month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns, y=pivot.index,
    colorscale=[[0, '#d73027'], [0.5, '#ffffbf'], [1, '#1a9850']],
    text=pivot.round(1).values, texttemplate='%{text}%',
))
fig.update_layout(**base_layout(title=f'{ticker_for_heatmap} — Monthly Return Heatmap (%)', height=400))
fig.show()

In [ ]:
# ── 10. Sector Composition & Risk Profile ────────────────────────────────────
sector_map = {
    'RELIANCE.NS': 'Energy',    'TCS.NS': 'IT',
    'HDFCBANK.NS': 'Banking',   'INFY.NS': 'IT',
    'ICICIBANK.NS': 'Banking',  'WIPRO.NS': 'IT',
    'BAJFINANCE.NS': 'NBFC',    'ASIANPAINT.NS': 'Consumer',
    'TITAN.NS': 'Consumer',     'MARUTI.NS': 'Auto',
    'ONGC.NS': 'Energy',        'ZOMATO.NS': 'Consumer Tech',
}
sector_vol = {s: returns[[t for t in TICKERS if sector_map[t]==s]].mean(axis=1).std() * np.sqrt(252) * 100
              for s in set(sector_map.values())}

fig = go.Figure(go.Bar(
    x=list(sector_vol.keys()), y=list(sector_vol.values()),
    marker_color=ORANGE, text=[f'{v:.1f}%' for v in sector_vol.values()],
    textposition='outside',
))
fig.update_layout(**base_layout(title='Annualised Volatility by Sector (%)', height=400))
fig.show()

print('\n✅  EDA complete — key findings:')
print(f'  • Highest avg vol:  {returns.std().idxmax().replace(".NS","")} at {returns.std().max()*np.sqrt(252)*100:.1f}% ann.')
print(f'  • Lowest drawdown:  {dd_stats["Max Drawdown (%)"].idxmax().replace(".NS","")} at {dd_stats["Max Drawdown (%)"].max():.1f}%')
print(f'  • Highest corr pair: {pairs.index[0]}')